# 📊 Notebook 1 — Exploratory Data Analysis
**Auto Insurance Churn Project**

### Goals
- Understand the dataset shape, quality, and churn distribution
- Identify the strongest visual signals correlated with churn
- Surface any data quality issues before modeling

### What is EDA?
EDA (Exploratory Data Analysis) is the detective phase of a data project.
Before building any model, we need to understand what the data actually looks like:
- How many customers? How many churned?
- Are there missing values we need to handle?
- What features visually separate churners from non-churners?

Good EDA drives better feature engineering and model decisions downstream.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

from data_loader import load_main

# Load 100K rows for fast EDA — swap nrows=None for full 1.68M dataset
df = load_main(nrows=100_000)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
df.head()


## 1. Dataset Overview

In [ ]:
# Missing values — critical to identify before modeling
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct}).query("`Missing Count` > 0").sort_values("Missing %", ascending=False)


## 2. Churn Distribution

**Key question:** How imbalanced is the dataset?

If churn is rare (< 15%), a naive model predicting 'never churn' achieves 88%+ accuracy
with zero business value. We need to see this before choosing our modeling approach.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df["churn"].value_counts()
axes[0].bar(["Retained", "Churned"], churn_counts.values, color=["steelblue", "tomato"])
axes[0].set_title("Churn Count")
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

churn_rate = df["churn"].mean() * 100
axes[1].pie(churn_counts.values, labels=["Retained", "Churned"],
            autopct="%1.1f%%", colors=["steelblue", "tomato"], startangle=90)
axes[1].set_title(f"Churn Rate: {churn_rate:.1f}%")

plt.suptitle("Churn Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/01_churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Churn by Tenure

**Hypothesis:** Newer customers are more likely to churn — they haven't built loyalty yet.

In [ ]:
bins = [0, 365, 1095, 2190, float("inf")]
labels = ["< 1yr", "1-3yr", "3-6yr", "6yr+"]
df["tenure_bucket"] = pd.cut(df["days_tenure"], bins=bins, labels=labels)

tenure_churn = df.groupby("tenure_bucket", observed=True)["churn"].agg(["mean", "count"]).reset_index()
tenure_churn["Churn Rate %"] = (tenure_churn["mean"] * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(tenure_churn["tenure_bucket"], tenure_churn["Churn Rate %"], color="tomato")
ax.set_title("Churn Rate by Customer Tenure", fontsize=13, fontweight="bold")
ax.set_xlabel("Tenure Bucket"); ax.set_ylabel("Churn Rate (%)")
for bar, val in zip(bars, tenure_churn["Churn Rate %"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{val}%", ha="center")
plt.tight_layout()
plt.savefig("../outputs/figures/01_churn_by_tenure.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Financial Profile vs Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

df[df["churn"]==0]["income"].plot(kind="kde", ax=axes[0], label="Retained", color="steelblue")
df[df["churn"]==1]["income"].plot(kind="kde", ax=axes[0], label="Churned", color="tomato")
axes[0].set_title("Income Distribution"); axes[0].legend()

credit_churn = df.groupby("good_credit")["churn"].mean() * 100
axes[1].bar(["No Good Credit", "Good Credit"], credit_churn.values, color=["tomato","steelblue"])
axes[1].set_title("Churn Rate by Credit")
for i, v in enumerate(credit_churn.values):
    axes[1].text(i, v + 0.1, f"{v:.1f}%", ha="center")

home_churn = df.groupby("home_owner")["churn"].mean() * 100
axes[2].bar(["Renter", "Homeowner"], home_churn.values, color=["tomato","steelblue"])
axes[2].set_title("Churn Rate by Home Ownership")
for i, v in enumerate(home_churn.values):
    axes[2].text(i, v + 0.1, f"{v:.1f}%", ha="center")

plt.suptitle("Financial Profile vs Churn", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/01_financial_profile_churn.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Correlation Heatmap

Which numeric features are most correlated with churn? This guides feature selection.

In [ ]:
numeric_cols = ["curr_ann_amt","days_tenure","age_in_years","income",
                "has_children","length_of_residence","home_owner",
                "college_degree","good_credit","churn"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/01_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## EDA Summary

| Finding | Implication for Modeling |
|---|---|
| ~12% churn rate | Use SMOTE or class weighting |
| Shorter tenure = higher churn | Tenure is a top feature |
| Higher premiums = more churn | Price sensitivity signal |
| Good credit = lower churn | Financial stability predictor |
| Renters churn more | Home ownership as stability proxy |

**Next:** `02_feature_engineering.ipynb`
